In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.patches import Rectangle
import rpy2.situation
for row in rpy2.situation.iter_info():
    print(row)
from rpy2.robjects import r
from rpy2.robjects.packages import importr
%matplotlib qt
from local_functions import pix_to_deg,read_data,calculate_bf,plot_bfs,shrink_axes,make_subplots, inset_axes


import rpy2.robjects as ro
ro.r('.libPaths(c(.libPaths(), "/Library/Frameworks/R.framework/Versions/4.6/Resources/library"))')
bf_package=importr('BayesFactor')

# A4 dimensions in inches
a4_width = 8.27 
a4_height = 11.69 

task = 'exp1a'

subj_colors = sns.color_palette("Spectral",21)

### read data

In [ ]:
bids_dir = '/Volumes/LaCie2/2021_Occlusion_behavioural/data'
subjids = [i for i in np.arange(1,21) if i!=9]
all_epochs = read_data(bids_dir,subjids,task='exp1a')

### Make plot

In [ ]:
### SETUP
colors = sns.color_palette('mako',7)
plt.close('all')
fig,axs = plt.subplots(2,2,num=1,figsize=(a4_width, a4_height*0.7))

# Labels for the subplots
labels = ['(a)', '(b)', '(c)','(d)']
for i, (ax, label) in enumerate(zip(axs.flat, labels)):
    pos = ax.get_position()  
    x = pos.x0 - 0.05
    y = pos.y1 + 0.02 
    fig.text(x, y, label, fontsize=12, fontweight='bold', va='top', ha='right')


#************************************
### (A) METHODS
#************************************
ax = shrink_axes(axs[0][0])
pos = ax.get_position()  
ax.set_position([pos.x0, pos.y0 + 0.1, pos.width, pos.height * 0.8]) 

epos = all_epochs[0].copy()
traj = np.linspace(-12.553666,12.553666,epos.metadata['n_frames'].unique()[0])

# figuring out timepoints when the object stopped moving / sound stopped playing
total_traj_time = epos.metadata['n_frames'].unique()[0]*1/epos.metadata['monitor_refreshrate'].unique()[0]
traj_tv = np.arange(0,total_traj_time,1/epos.metadata['monitor_refreshrate'].unique()[0])
stopping_idx_traj = [np.argmin(np.abs(traj_tv-v)) for v in epos.metadata.stop_movement_time]

ax.plot(traj_tv[(traj>=-4.5)&(traj<=4.5)],traj[(traj>=-4.5)&(traj<=4.5)],':',color='k',lw=1.5)
ax.plot(traj_tv[(traj<-4.5)],traj[(traj<-4.5)],'-',color='k',lw=1.5)
ax.plot(traj_tv[(traj>=4.5)],traj[(traj>=4.5)],'-',color='k',lw=0)

# plot stopping positions
dat = all_epochs[0].metadata.copy()
dat.loc[dat.direction==2,'stop_pos_deg'] = dat.loc[dat.direction==2,'stop_pos_deg']*-1
dat.loc[dat.direction==2,'stop_pos'] = dat.loc[dat.direction==2,'stop_pos']*-1
dat.loc[dat.direction==2,'pos_pressed_deg'] = dat.loc[dat.direction==2,'pos_pressed_deg']*-1
dat.loc[dat.direction==2,'pos_pressed'] = dat.loc[dat.direction==2,'pos_pressed']*-1
x = np.sort(dat['stop_movement_time'].unique())
true_y = dat.groupby('stop_movement_time').mean(numeric_only=True).stop_pos_deg.to_numpy()

[ax.plot(x[i], true_y[i], marker='*', color=colors[i], alpha=0.6,markersize=10, lw=0, markeredgecolor=None) for i in range(7)]

# Annotate stopping positions
branch_x = 1.85
branch_y = -0.5
[ax.annotate('', xy=(x[i], true_y[i]),xytext=(branch_x, branch_y),arrowprops=dict(arrowstyle='->', color='k', lw=0.8, shrinkB=4),clip_on=False) for i in range(7)]
ax.text(branch_x+0.1, branch_y-0.1, 'stopping positions',fontsize=9, rotation=42, ha='center', va='center', clip_on=False)


# plot occluder annotation
x_pos = 0.6
ax.vlines(x_pos,-np.max(epos.metadata.occluder_start_pos_deg),np.max(epos.metadata.occluder_start_pos_deg),'k',linestyles='-',lw=2)
ax.hlines(-np.max(epos.metadata.occluder_start_pos_deg),x_pos,x_pos+0.1,'k',linestyles='-',lw=2)
ax.hlines(np.max(epos.metadata.occluder_start_pos_deg),x_pos,x_pos+0.1,'k',linestyles='-',lw=2)
ax.annotate('occluder', xy=(x_pos, 5),xytext=(x_pos, 10),ha='left', va='top', fontsize=9,arrowprops=dict(arrowstyle='->', color='k'),clip_on=False)

ax.set_ylim([-13,13])
ax.set_xlabel('time since trial start (s)')
ax.set_ylabel('x-position object (dva)')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

#************************************
### (B) BEHAVIOURAL PERFORMANCE
#************************************
ax = shrink_axes(axs[0][1])

# stack group data
group_data = []
for s in range(len(all_epochs)):
    dat = all_epochs[s].metadata.copy()
    # ignoring different motion directions [L->R and R->L]
    dat.loc[dat.direction==2,'stop_pos_deg'] = dat.loc[dat.direction==2,'stop_pos_deg']*-1
    dat.loc[dat.direction==2,'stop_pos'] = dat.loc[dat.direction==2,'stop_pos']*-1
    dat.loc[dat.direction==2,'pos_pressed_deg'] = dat.loc[dat.direction==2,'pos_pressed_deg']*-1
    dat.loc[dat.direction==2,'pos_pressed'] = dat.loc[dat.direction==2,'pos_pressed']*-1

    x = np.sort(dat['stop_movement_time'].unique())
    y = dat.groupby('stop_movement_time').mean(numeric_only=True).pos_pressed_deg.to_numpy()
    ax.plot(x,y,'k.',alpha=0.5)
    
    group_data.append(y)

# plot true positions and individual selections 
true_y = dat.groupby('stop_movement_time').mean(numeric_only=True).stop_pos_deg.to_numpy()
[ax.plot(x[i],true_y[i],marker='*',color=colors[i],alpha=0.6, markersize=10,lw=0,markeredgecolor=None) for i in range(7)]

# style
ax.set_ylim([-5,8])
ax.set_yticks(np.arange(-3,8,2))
ax.set_xticks(x)
ax.set_xticklabels('')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
label = ax.set_ylabel('Relative position\n clicked (dva)')
label.set_position((label.get_position()[0],label.get_position()[1]*1.2))
pos = ax.get_position()  
ax.set_position([pos.x0, pos.y0 + 0.1, pos.width, pos.height * 0.8]) 

# add Bayes factors
ax_in = inset_axes(ax,
                width="100%",  # Width of inset as a percentage of axs[1][0] width
                height="20%",  # Height of inset as a percentage of axs[1][0] height
                bbox_to_anchor=(0.0, -0.3, 1.0, 1.0),  # Relative position below axs[1][0]
                bbox_transform=ax.transAxes,  # Transform to use axs[1][0]'s coordinate system
                loc='lower center') 

mean_beh_error = np.array(group_data)-true_y
bf_out_beh = calculate_bf(mean_beh_error)
exponential_minmax=12
plot_bfs(fig,ax_in,x=x,y=bf_out_beh,n_cols = 500,bf_cmap="coolwarm",exponential_minmax=exponential_minmax)
ax_in.set_xlabel('time since trial start (s)')
ax_in.set_xticklabels([np.around(i,2) for i in x])
[ax_in.spines[i].set_visible(False) for i in ['top','right']]
ax_in.set_ylabel('BF',fontsize=12)
ax_in.tick_params(axis='y', which='major', labelsize=8)

#************************************
### (C) EYETRACKING
#************************************

### Show data for one participant and one trial type (i), for one participant and all trial types (ii) and for everyone averaged (iii)
sample_p = 0
epos = all_epochs[sample_p].copy()
x_idx = np.where([i=='xpos' for i in epos.ch_names])[0][0]
epos._data[epos.metadata.direction==2,x_idx,:] = epos._data[epos.metadata.direction==2,x_idx,:]*-1

unique_stop_movement_time  = np.sort(epos.metadata.stop_movement_time.unique())

traj = np.linspace(-12.553666,12.553666,epos.metadata['n_frames'].unique()[0])
total_traj_time = epos.metadata['n_frames'].unique()[0]*1/epos.metadata['monitor_refreshrate'].unique()[0]
traj_tv = np.arange(0,total_traj_time,1/epos.metadata['monitor_refreshrate'].unique()[0])


## (i)
ax = axs[1][0]
ax_in = make_subplots(ax,bottom=0.7)
ax_in.plot(traj_tv,traj,'k:',label='object trajectory',lw=1)

idx = 3 # take one trial type as example
v  = unique_stop_movement_time[idx]
    
# add marker for stopping position and connector lines
stopping_idx_traj = np.argmin(np.abs(traj_tv-v))
ax_in.plot(traj_tv[stopping_idx_traj],traj[stopping_idx_traj],marker='*',color=colors[i],markeredgecolor=None,markersize=10,alpha=0.6)
stopping_idx_epo = (epos.times<=0).sum()+np.argmin(np.abs(epos.times[(epos.times>0)]-traj_tv[stopping_idx_traj]))

# plot mean x-gaze 
sel_epo = epos[(epos.metadata.stop_movement_time==v)]
toplot = np.nanmean(sel_epo._data[:,x_idx],axis=0)

ax_in.vlines(epos.times[stopping_idx_epo],toplot[stopping_idx_epo],traj[stopping_idx_traj],color=colors[idx],linestyles=':')
ax_in.plot(sel_epo.times[:stopping_idx_epo],toplot[:stopping_idx_epo],color = colors[idx],lw=2,alpha=0.8)

# plot individual trials
[ax_in.plot(sel_epo.times[:stopping_idx_epo],ts[:stopping_idx_epo],color = colors[idx],alpha=0.5,lw=0.5) for ts in sel_epo._data[:,x_idx]]
ax_in.vlines(epos.metadata['occlusion_start_time'].unique()[0],-4.5,4.5,color='k')

# style axis
ax_in.spines['right'].set_visible(False)
ax_in.spines['top'].set_visible(False)
ax_in.set_xlim(0,epos.times[-1])
ax_in.set_xticklabels('')
ax_in.text(0.1, 0.98, f'(i) sub-S{str(sample_p+1).zfill(2)}', 
    transform=ax.transAxes, fontsize=12, verticalalignment='top', horizontalalignment='left'
)

# (ii)
ax_in = make_subplots(ax,bottom=0.35)

ax_in.plot(traj_tv,traj,'k:',label='object trajectory',lw=1)

for i,v in enumerate(unique_stop_movement_time):
    # plot mean x-gaze
    sel_epo = epos[(epos.metadata.stop_movement_time==v)]
    toplot = np.nanmean(sel_epo._data[:,x_idx],axis=0)
    stopping_idx_traj = np.argmin(np.abs(traj_tv-v))
    stopping_idx_epo = (epos.times<=0).sum()+np.argmin(np.abs(epos.times[(epos.times>0)]-traj_tv[stopping_idx_traj]))
    ax_in.plot(sel_epo.times[:stopping_idx_epo],toplot[:stopping_idx_epo],color = colors[i],lw=1,alpha=0.8)
for i,v in enumerate(unique_stop_movement_time):    
    # add marker for stopping position and connector lines
    stopping_idx_traj = np.argmin(np.abs(traj_tv-v))
    ax_in.plot(traj_tv[stopping_idx_traj],traj[stopping_idx_traj],marker='*',color=colors[i],markeredgecolor=None,markersize=10,alpha=0.6)
    stopping_idx_epo = (epos.times<=0).sum()+np.argmin(np.abs(epos.times[(epos.times>0)]-traj_tv[stopping_idx_traj]))
    ax_in.vlines(epos.times[stopping_idx_epo],toplot[stopping_idx_epo],traj[stopping_idx_traj],color=colors[i],linestyles=':')
    
ax_in.vlines(epos.metadata['occlusion_start_time'].unique()[0],-4.5,4.5,color='k')

# style axis
ax_in.set_ylabel('x-gaze position (dva)')
ax_in.spines['right'].set_visible(False)
ax_in.spines['top'].set_visible(False)
ax_in.set_xlim(0,epos.times[-1])
ax_in.set_xticklabels('')
ax_in.text(0.1, 0.63, f'(ii) sub-S{str(sample_p+1).zfill(2)}', 
    transform=ax.transAxes, fontsize=12, verticalalignment='top', horizontalalignment='left'
)

# (iii)
ax_in = make_subplots(ax,bottom=0)
ax_in.plot(traj_tv,traj,'k:',label='object trajectory',lw=1)

for i,v in enumerate(unique_stop_movement_time):
    # plot mean x-gaze
    toplot_all = []
    for p in range(len(subjids)):
        epos = all_epochs[p].copy()
        x_idx = np.where([i=='xpos' for i in epos.ch_names])[0][0]
        epos._data[epos.metadata.direction==2,x_idx,:] = epos._data[epos.metadata.direction==2,x_idx,:]*-1
        
        sel_epo = epos[(epos.metadata.stop_movement_time==v)]
        toplot_all.append(np.nanmean(sel_epo._data[:,x_idx],axis=0))
    toplot = np.mean(toplot_all,axis=0)
    stopping_idx_traj = np.argmin(np.abs(traj_tv-v))
    stopping_idx_epo = (epos.times<=0).sum()+np.argmin(np.abs(epos.times[(epos.times>0)]-traj_tv[stopping_idx_traj]))
    ax_in.plot(sel_epo.times[:stopping_idx_epo],toplot[:stopping_idx_epo],color = colors[i],lw=1,alpha=0.8)


gaze_error = []
for i,v in enumerate(unique_stop_movement_time):    
    # add marker for stopping position and connector lines
    stopping_idx_traj = np.argmin(np.abs(traj_tv-v))
    ax_in.plot(traj_tv[stopping_idx_traj],traj[stopping_idx_traj],marker='*',color=colors[i],markeredgecolor=None,markersize=10,alpha=0.6)
    stopping_idx_epo = (epos.times<=0).sum()+np.argmin(np.abs(epos.times[(epos.times>0)]-traj_tv[stopping_idx_traj]))
    ax_in.vlines(epos.times[stopping_idx_epo],toplot[stopping_idx_epo],traj[stopping_idx_traj],color=colors[i],linestyles=':')
    
    gaze_error.append((np.array(toplot_all)[:,stopping_idx_epo])-traj[stopping_idx_traj])

ax_in.vlines(epos.metadata['occlusion_start_time'].unique()[0],-4.5,4.5,color='k')

# style axis
ax_in.spines['right'].set_visible(False)
ax_in.spines['top'].set_visible(False)
ax_in.set_xlabel('time since trial start (s)')
ax_in.set_xlim(0,epos.times[-1])
ax_in.text(0.1, 0.28, f'(iii) Average (n={len(subjids)})', 
    transform=ax.transAxes, fontsize=12, verticalalignment='top', horizontalalignment='left'
)

ax.set_visible(False)

#************************************
### (D) EYETRACKING & BEHAVIOUR CORRELATION
#************************************
ax = shrink_axes(axs[1][1])
all_corr = []
for p in range(len(all_epochs)):

    epos = all_epochs[p].copy()

    # ignoring different motion directions [L->R and R->L]
    epos.metadata.loc[epos.metadata.direction==2,'stop_pos_deg'] = epos.metadata.loc[epos.metadata.direction==2,'stop_pos_deg']*-1
    epos.metadata.loc[epos.metadata.direction==2,'stop_pos'] = epos.metadata.loc[epos.metadata.direction==2,'stop_pos']*-1
    epos.metadata.loc[epos.metadata.direction==2,'pos_pressed_deg'] = epos.metadata.loc[epos.metadata.direction==2,'pos_pressed_deg']*-1
    epos.metadata.loc[epos.metadata.direction==2,'pos_pressed'] = epos.metadata.loc[epos.metadata.direction==2,'pos_pressed']*-1
    epos._data[epos.metadata.direction==2,x_idx,:] = epos._data[epos.metadata.direction==2,x_idx,:]*-1

    # calculate behavioural error
    beh_error = epos.metadata.pos_pressed_deg-epos.metadata.stop_pos_deg

    # figuring out timepoints when the object stopped moving / sound stopped playing
    total_traj_time = epos.metadata['n_frames'].unique()[0]*1/epos.metadata['monitor_refreshrate'].unique()[0]
    traj_tv = np.arange(0,total_traj_time,1/epos.metadata['monitor_refreshrate'].unique()[0])
    stopping_idx_traj = [np.argmin(np.abs(traj_tv-v)) for v in epos.metadata.stop_movement_time]
    stopping_idx_epo = [(epos.times<=0).sum()+np.argmin(np.abs(epos.times[(epos.times>0)]-traj_tv[v])) for v in stopping_idx_traj]

    # calculate eyetracking error
    x_idx = np.where([i=='xpos' for i in epos.ch_names])[0][0]
    eye_error = [epos._data[i,x_idx,v] for i,v in enumerate(stopping_idx_epo)]-epos.metadata.stop_pos_deg.to_numpy()

    # plot
    ax.plot(beh_error,eye_error,marker='.',color=subj_colors[p],lw=0,alpha=0.4,markeredgecolor=None)
    
    all_corr.append(pd.DataFrame(np.array([beh_error,eye_error]).T,columns=['beh','eye']).corr().iloc[0,1])

ax.plot(np.linspace(-10,10,100),np.linspace(-10,10,100),'k--')
# style axis
ax.set_ylabel('error gaze position (dva)')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.set_xlabel('error behavior (dva)')
ax.set_xlim(-10,10)
ax.set_ylim(-10,10)

bf_out = calculate_bf(np.expand_dims(all_corr,axis=1))
print(f'mean correlation behaviour and gaze error  = {np.mean(all_corr)}, BF = 10^{np.log10(np.array(bf_out))}')


#************************************
### SAVE
#************************************
plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp1a.png',dpi=300)

plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp1a.pdf')


### Bayes Factor tables


In [ ]:
def make_table(df,fn):
    fig, ax = plt.subplots(figsize=(a4_width, a4_height*0.3))
    fig.patch.set_visible(False)
    ax.axis('off')
    ax.axis('tight')

    table=ax.table(cellText=df.values,colLabels= df.columns, rowLabels=df.index.astype(str), loc='center')

    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 2)
    
    plt.tight_layout()
    plt.savefig(f"{bids_dir}/{task}/derivatives/figures/{fn}.pdf", bbox_inches='tight')
    
# gaze error
bf_out_gaze = calculate_bf(np.array(gaze_error).T)
col_names = [f"$\\mathbf{{stop\ {i}}}$\n$({v:.2f}s)$" for i, v in enumerate(unique_stop_movement_time)]
mean_error = np.round(np.mean(np.array(gaze_error),axis=1),2)
bf_log = np.array([[f"$10^{{{x:.2f}}}$"] for x in np.log10(np.array(bf_out_gaze))]).reshape(1,-1)
print(bf_out_gaze)

table_data = np.vstack((mean_error,bf_log))
df = pd.DataFrame(table_data, columns=col_names, index=['gaze error (dva)','Bayes Factor'])

make_table(df,'bfs_gaze_error')

# beh error
col_names = [f"$\\mathbf{{stop\ {i}}}$\n$({v:.2f}s)$" for i, v in enumerate(x)]
mean_error = np.round(np.mean(mean_beh_error,axis=0),2)
print(bf_out_beh)
bf_log = np.array([[f"$10^{{{x:.2f}}}$"] for x in np.log10(np.array(bf_out_beh))]).reshape(1,-1)
table_data = np.vstack((mean_error,bf_log))
df = pd.DataFrame(table_data, columns=col_names, index=['behavioral error (dva)','Bayes Factor'])

make_table(df,'bfs_beh_error')